In [52]:
import nltk
import re
import pandas as pd

from nltk.corpus import movie_reviews
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

print("Libraries imported successfully!")

Libraries imported successfully!


In [53]:
nltk.download('movie_reviews')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [54]:
# Get all movie review file IDs
review_ids = movie_reviews.fileids()

# Extract review texts
reviews = [
    movie_reviews.raw(fileid)
    for fileid in review_ids
]

# Extract review categories
labels = [
    movie_reviews.categories(fileid)[0]
    for fileid in review_ids
]

# Create DataFrame
df = pd.DataFrame({
    'review': reviews,
    'sentiment': labels
})

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset shape: (2000, 2)


,review,sentiment
0,"plot : two teen couples go to a church party ,...",neg
1,the happy bastard's quick movie review \ndamn ...,neg
2,it is movies like these that make a jaded movi...,neg
3,""" quest for camelot "" is warner bros . ' firs...",neg
4,synopsis : a mentally unstable man undergoing ...,neg


In [55]:
# Load English stop words
stop_words = set(stopwords.words('english'))


stop_words -= {'not', 'no', 'nor'}

print("Stop words loaded:", len(stop_words))

Stop words loaded: 195


In [56]:
def preprocess_text(text):
    text = text.lower()

    # Tokenize text
    tokens = word_tokenize(text)

    cleaned_tokens = []

    for token in tokens:
        # Preserve negation contractions
        if token == "n't":
            cleaned_tokens.append("not")

        elif token.isalpha() and token not in stop_words:
            cleaned_tokens.append(token)

    return cleaned_tokens

In [57]:
# Apply preprocessing to reviews
df['tokens'] = df['review'].apply(preprocess_text)

# Create cleaned text
df['cleaned_review'] = df['tokens'].apply(
    lambda tokens: ' '.join(tokens)
)

print("Text preprocessing completed!")

df[['review', 'tokens', 'cleaned_review']].head()

Text preprocessing completed!


,review,tokens,cleaned_review
0,"plot : two teen couples go to a church party ,...","[plot, two, teen, couples, go, church, party, ...",plot two teen couples go church party drink dr...
1,the happy bastard's quick movie review \ndamn ...,"[happy, bastard, quick, movie, review, damn, b...",happy bastard quick movie review damn bug got ...
2,it is movies like these that make a jaded movi...,"[movies, like, make, jaded, movie, viewer, tha...",movies like make jaded movie viewer thankful i...
3,""" quest for camelot "" is warner bros . ' firs...","[quest, camelot, warner, bros, first, attempt,...",quest camelot warner bros first attempt steal ...
4,synopsis : a mentally unstable man undergoing ...,"[synopsis, mentally, unstable, man, undergoing...",synopsis mentally unstable man undergoing psyc...


In [58]:
# Select the first review
original_text = df.loc[0, 'review']
processed_tokens = df.loc[0, 'tokens']

print("Original review:")
print(original_text[:500])

print("\nProcessed tokens:")
print(processed_tokens[:50])

print("\nNumber of tokens:", len(processed_tokens))

Original review:
plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the guys dies , but his girlfriend continues to see him in her life , and has nightmares . 
what's the deal ? 
watch the movie and " sorta " find out . . . 
critique : a mind-fuck movie for the teen generation that touches on a very cool idea , but presents it in a very bad package . 
which is what makes this review an even harder one to write , since i generally applaud films which attempt

Processed tokens:
['plot', 'two', 'teen', 'couples', 'go', 'church', 'party', 'drink', 'drive', 'get', 'accident', 'one', 'guys', 'dies', 'girlfriend', 'continues', 'see', 'life', 'nightmares', 'deal', 'watch', 'movie', 'sorta', 'find', 'critique', 'movie', 'teen', 'generation', 'touches', 'cool', 'idea', 'presents', 'bad', 'package', 'makes', 'review', 'even', 'harder', 'one', 'write', 'since', 'generally', 'applaud', 'films', 'attempt', 'break', 'mold', 'mess', 'head', 'lost

In [59]:
# Calculate the number of tokens in reviews
df['token_count'] = df['tokens'].apply(len)

print("Total reviews:", len(df))
print("Total tokens:", df['token_count'].sum())
print("Average tokens per review:", round(
    df['token_count'].mean(), 2
))

print("\nSentiment distribution:")
print(df['sentiment'].value_counts())

print("\nDataset preview:")
df[['sentiment', 'cleaned_review', 'token_count']].head()

Total reviews: 2000
Total tokens: 693591
Average tokens per review: 346.8

Sentiment distribution:
sentiment
neg    1000
pos    1000
Name: count, dtype: int64

Dataset preview:


,sentiment,cleaned_review,token_count
0,neg,plot two teen couples go church party drink dr...,328
1,neg,happy bastard quick movie review damn bug got ...,131
2,neg,movies like make jaded movie viewer thankful i...,268
3,neg,quest camelot warner bros first attempt steal ...,251
4,neg,synopsis mentally unstable man undergoing psyc...,385


# And now, it`s Vectorin' time!



In [60]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("TF-IDF Vectorizer imported successfully!")

TF-IDF Vectorizer imported successfully!


In [61]:
# Initialize  Vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000
)

# Convert cleaned reviews into  vectors
X = tfidf_vectorizer.fit_transform(df['cleaned_review'])

print("Vectorization completed successfully!")
print("Feature matrix shape:", X.shape)

Vectorization completed successfully!
Feature matrix shape: (2000, 10000)


In [62]:
# Get feature names
feature_names = tfidf_vectorizer.get_feature_names_out()

print("Total number of features:", len(feature_names))

print("\nFirst 30 features:")
print(feature_names[:30])

Total number of features: 10000

First 30 features:
['aaron' 'abandon' 'abandoned' 'abandons' 'abby' 'aberdeen' 'abilities'
 'ability' 'able' 'ably' 'aboard' 'abortion' 'abound' 'abraham' 'abrasive'
 'abrupt' 'abruptly' 'absence' 'absent' 'absolute' 'absolutely' 'absorb'
 'absorbed' 'absorbing' 'abstract' 'absurd' 'absurdity' 'abundance'
 'abuse' 'abused']


In [63]:
# Convert the first five vectors into a DataFrame
vectors_preview = pd.DataFrame(
    X[:5].toarray(),
    columns=feature_names
)

print("Numerical representation of the first five reviews:")
vectors_preview.iloc[:, :20]

Numerical representation of the first five reviews:


,aaron,abandon,abandoned,abandons,abby,aberdeen,abilities,ability,able,ably,aboard,abortion,abound,abraham,abrasive,abrupt,abruptly,absence,absent,absolute
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.038616,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [64]:
# Extract target labels
y = df['sentiment']

print("Target labels prepared successfully!")
print("\nLabel distribution:")
print(y.value_counts())

print("\nFeature matrix:", X.shape)
print("Target vector:", y.shape)   

Target labels prepared successfully!

Label distribution:
sentiment
neg    1000
pos    1000
Name: count, dtype: int64

Feature matrix: (2000, 10000)
Target vector: (2000,)


# Logic Registration

In [65]:
from sklearn.linear_model import LogisticRegression

print("Logistic Regression imported successfully!")

Logistic Regression imported successfully!


In [66]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

print("Logistic Regression model created!")

Logistic Regression model created!


In [67]:
from sklearn.pipeline import Pipeline

print("Pipeline imported successfully!")

Pipeline imported successfully!


In [68]:
# Creating a pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000)),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

print(" Pipeline created successfully!")

 Pipeline created successfully!


# Cross-Validation

In [69]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("Cross-validation tools imported successfully!")

Cross-validation tools imported successfully!


In [70]:
# Initialize  cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Stratified Cross-Validation configured!")

Stratified Cross-Validation configured!


In [71]:
# Perform 5-fold cross-validation
X_text = df['cleaned_review']
y = df['sentiment']

cv_scores = cross_val_score(
    pipeline,
    X_text,
    y,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

print("Cross-validation completed successfully!")

Cross-validation completed successfully!


In [72]:
print("Cross-Validation Results:\n")

for fold, score in enumerate(cv_scores, start=1):
    print(f"Fold {fold}: {score:.4f} ({score * 100:.2f}%)")

Cross-Validation Results:

Fold 1: 0.8500 (85.00%)
Fold 2: 0.8100 (81.00%)
Fold 3: 0.8325 (83.25%)
Fold 4: 0.8375 (83.75%)
Fold 5: 0.8325 (83.25%)


In [73]:
# Calculate average accuracy and standard deviation
mean_accuracy = cv_scores.mean()
std_accuracy = cv_scores.std()

print("FINAL MODEL EVALUATION")
print(f"Mean Accuracy: {mean_accuracy:.4f}")
print(f"Mean Accuracy (%): {mean_accuracy * 100:.2f}%")
print(f"Standard Deviation: {std_accuracy:.4f}")

FINAL MODEL EVALUATION
Mean Accuracy: 0.8325
Mean Accuracy (%): 83.25%
Standard Deviation: 0.0129
